# `rating.py` Reference

Rating functions: `(agent_seq) → float`. All take a sequence of
`(agents_for, agents_against)` tuples and return a numeric rating.
Weighted variants also accept a parallel `weights` sequence.
These functions plug directly into `make_rank_key` / `sort_groups` as `rating_fn`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

In [ ]:
from tournament.generators import make_players, skilled_match, random_match
from tournament.engine import run_tournament
from tournament.pairing import get as get_pairing

rng = Random(42)
players = make_players(8, rng)
tour = run_tournament(players, n_rounds=4, pairing=get_pairing('adjacent'), rng=rng)

In [ ]:
from tournament.standings import compute_records
records = compute_records(tour)

# Pick two players with different records to illustrate
best_pid  = max(records, key=lambda p: records[p].wins)
worst_pid = min(records, key=lambda p: records[p].wins)
seq_best  = records[best_pid].agent_seq
seq_worst = records[worst_pid].agent_seq
print(f'Best  ({players[best_pid].name}):  seq={seq_best}')
print(f'Worst ({players[worst_pid].name}): seq={seq_worst}')

## Basic rating functions

In [ ]:
from tournament.rating import (
    total_agents_scored, total_agents_lost, agent_differential,
    agent_ratio, agent_total_ratio,
)

print(f'{"Function":<25} {"best":>10} {"worst":>10}')
print('-' * 47)
for name, fn in [
    ('total_agents_scored',  total_agents_scored),
    ('total_agents_lost',    total_agents_lost),
    ('agent_differential',   agent_differential),
    ('agent_ratio',          agent_ratio),
    ('agent_total_ratio',    agent_total_ratio),
]:
    b, w = fn(seq_best), fn(seq_worst)
    print(f'{name:<25} {b:>10.3f} {w:>10.3f}')

## Edge cases: zero denominators

In [ ]:
perfect   = [(3, 0), (3, 0), (3, 0)]   # never lost an agent
no_data   = []                           # no matches played

print(f'agent_ratio(perfect):        {agent_ratio(perfect)}')
print(f'agent_ratio(no_data):        {agent_ratio(no_data)}')
print(f'agent_total_ratio(perfect):  {agent_total_ratio(perfect):.3f}')
print(f'agent_total_ratio(no_data):  {agent_total_ratio(no_data):.3f}')

## Weight utilities

In [ ]:
from tournament.rating import linear_weights, exponential_weights

for n in (3, 5):
    lin = linear_weights(n, 1.0, 2.0)
    exp = exponential_weights(n, base=1.5)
    print(f'n={n}  linear(1→2): {[round(w,3) for w in lin]}')
    print(f'     exponential(x1.5): {[round(w,3) for w in exp]}')

## Weighted rating functions

In [ ]:
from tournament.rating import (
    weighted_total_agents_scored, weighted_total_agents_lost,
    weighted_agent_differential, weighted_agent_ratio, weighted_agent_total_ratio,
)

n   = len(seq_best)
lin = linear_weights(n, 1.0, 2.0)

print(f'Weighted (linear 1→2) for {players[best_pid].name}  seq={seq_best}')
print(f'  weighted_total_agents_scored  = {weighted_total_agents_scored(seq_best, lin):.3f}')
print(f'  weighted_total_agents_lost    = {weighted_total_agents_lost(seq_best, lin):.3f}')
print(f'  weighted_agent_differential   = {weighted_agent_differential(seq_best, lin):.3f}')
print(f'  weighted_agent_ratio          = {weighted_agent_ratio(seq_best, lin):.3f}')
print(f'  weighted_agent_total_ratio    = {weighted_agent_total_ratio(seq_best, lin):.3f}')

## Using a rating function as a tiebreaker

In [ ]:
import functools
from tournament.standings import make_rank_key

recency = functools.partial(weighted_agent_differential,
                            weights=linear_weights(4, 1.0, 2.0))

key_plain   = make_rank_key()                   # agent_differential
key_recency = make_rank_key(rating_fn=recency)  # recent rounds count more

print(f'{"Name":<8} {"W":>3} {"plain":>8} {"recency":>10}')
print('-' * 32)
for r in sorted(records.values(), key=key_plain, reverse=True):
    print(f'{players[r.pid].name:<8} {r.wins:>3} '
          f'{key_plain(r):>8.3f} {key_recency(r):>10.3f}')